# Cleanup ghost trades → mark pre-#357 open rows as `orphaned`

**One-shot data migration.** Run this once after the rogue interpreter is dead and the bot is loading post-#357 code.

## What this does

Pre-PR #357, `_log_trade_to_journal` wrote `status='open'` to `trade_journal.db::trades` *before* the exchange call returned. If the exchange refused (insufficient balance, leverage cap, position-mode mismatch), the row was orphaned with no rejection-row counterpart. PR #357 prevents this for new trades but doesn't backfill existing ghosts.

Symptom: `/last5` shows trades with `status=open` that have no matching exchange position (operator's run on 2026-05-03 returned 5 such rows from 22:56 yesterday → 01:49 today, while `bybit_2` reported `open positions: 0`).

This notebook:
1. SSHes to the VM.
2. **Previews** every `status='open'` row whose `created_at` is before a configurable cutoff (default: `2026-05-03T16:00:00 UTC`, the rogue-kill time).
3. Pauses for you to set `CONFIRM = True` in Cell 4.
4. **Updates** matching rows to `status='orphaned'` (which is in `REFUSAL_STATUSES` so they get filtered from `/last5` and shown in `/packages`).

**Reversibility.** The migration only touches `status` and appends `orphaned_at` to the `notes` JSON. Inverse: `UPDATE trades SET status='open' WHERE status='orphaned'`.

**Safety.** Cell 2 is read-only; you see exactly what would change before Cell 4 runs anything.

## When to re-run

Only if a **new batch of ghost trades** appears (would indicate a regression in the dispatch/journal write order). The expected steady state is zero new ghost trades post-#357.

## Required Colab Secrets

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | VM hostname or public IP (same as the rotate-keys notebook) |
| `VM_SSH_USER` | SSH user on the VM (usually `ubuntu`) |

Plus the same SSH key file in `My Drive/ICT_Bot_Secrets/` that the rotate-keys notebook uses.


In [ ]:
# Cell 1: SSH setup (compact — mirrors rotate_api_keys.ipynb pattern).
import os, shutil, stat, subprocess, tempfile
from google.colab import drive, userdata

print('⏳ Mounting Google Drive (one-click "Allow" on first run per session)…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

DEFAULT_SSH_KEY_NAMES = [
    'ict-bot-ovm-private.key',
    'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa',
]
DRIVE_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'

ssh_key_path = None
if os.path.exists('/content/drive/MyDrive'):
    for name in DEFAULT_SSH_KEY_NAMES:
        path = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            break
if ssh_key_path is None:
    raise SystemExit(
        f'No SSH key found in {DRIVE_FOLDER}/. Place '
        '`ict-bot-ovm-private.key` (or any of '
        f'{DEFAULT_SSH_KEY_NAMES}) there and re-run.'
    )
print(f'✅ SSH key located: {ssh_key_path}')

host = userdata.get('VM_SSH_HOST')
user = userdata.get('VM_SSH_USER')
if not host or not user:
    raise SystemExit('VM_SSH_HOST + VM_SSH_USER must be set in Colab Secrets.')

tmpdir = tempfile.mkdtemp(prefix='cleanup-ghost-')
key_path = os.path.join(tmpdir, 'vm_key')
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]

def run_ssh(cmd, *, label, allow_fail=False):
    proc = subprocess.run(
        ['ssh'] + ssh_opts + [ssh_target, cmd],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print((proc.stderr or proc.stdout)[:500])
        if not allow_fail:
            raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()

print(f'⏳ Connecting to {ssh_target} …')
run_ssh('echo connected', label='SSH connectivity')
print('✅ SSH OK')


In [ ]:
# Cell 2: PREVIEW (read-only). Shows every row that WOULD be marked
# orphaned by Cell 4. Inspect the output before running Cell 4.

# Cutoff: anything created BEFORE this UTC timestamp is considered
# pre-rogue-kill / pre-#357 and a candidate for the ghost-trade
# backfill. Default = the rogue-kill time on 2026-05-03 + a small
# safety buffer. Edit if you have a different ground truth.
CUTOFF_UTC = '2026-05-03T16:30:00'

DB_PATH = f'/home/{user}/ict-trading-bot/trade_journal.db'

preview_sql = (
    f"SELECT id, created_at, account_id, strategy_name, symbol, direction, "
    f"position_size, status FROM trades "
    f"WHERE status = 'open' AND COALESCE(is_backtest, 0) = 0 "
    f"AND created_at < '{CUTOFF_UTC}' "
    f"ORDER BY datetime(created_at) DESC;"
)
rows = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{preview_sql}"',
    label='preview ghost trades',
)
print(f'Rows that WILL be marked orphaned (cutoff = {CUTOFF_UTC} UTC):')
print()
print(rows or '(no rows match — nothing to clean up)')

count = run_ssh(
    f'sqlite3 {DB_PATH} "SELECT COUNT(*) FROM trades '
    f"WHERE status = '\"'\"'open'\"'\"' AND COALESCE(is_backtest, 0) = 0 "
    f"AND created_at < '\"'\"'{CUTOFF_UTC}'\"'\"'\"",
    label='count candidates',
)
print(f'\nTotal: {count} row(s) would change.')


---

## ⚠️ Inspect Cell 2's preview above

If the rows look right — they're the ghost trades you want to retire — set `CONFIRM = True` in Cell 4 below and run it. Otherwise leave `CONFIRM = False` and Cell 4 will report what *would* have changed without actually changing anything.


In [ ]:
# Cell 4: COMMIT. Set CONFIRM = True only after inspecting Cell 2's output.
#
# IMPORTANT: this cell only cleans up the SSH-key tmpdir AFTER a
# successful UPDATE. A CONFIRM=False preview run leaves tmpdir
# intact so you can flip CONFIRM and re-run without re-running
# Cell 1 (which would re-prompt for Drive auth on a fresh session).
CONFIRM = False  # ← set to True to commit the UPDATE

if not CONFIRM:
    print('CONFIRM is False — no changes made. Set CONFIRM = True '
          'and re-run this cell to commit. (tmpdir kept alive so '
          'you do not need to re-run Cell 1.)')
else:
    # Set status='orphaned' and append orphaned_at marker to notes JSON
    # so the migration is auditable. notes is a TEXT column holding
    # JSON; we use SQLite json_set to insert the marker. If the
    # column is NULL we initialise with an empty object first.
    update_sql = (
        "UPDATE trades SET "
        "status = 'orphaned', "
        "notes = json_set(COALESCE(notes, '{}'), "
        "'$.orphaned_at', strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "'$.orphaned_by', 'cleanup_ghost_trades.ipynb', "
        "'$.orphaned_reason', 'pre-PR-357 ghost — logged status=open before exchange call returned') "
        f"WHERE status = 'open' AND COALESCE(is_backtest, 0) = 0 "
        f"AND created_at < '{CUTOFF_UTC}';"
    )
    affected = run_ssh(
        f'sqlite3 {DB_PATH} "{update_sql} SELECT changes();"',
        label='UPDATE ghost trades',
    )
    print(f'✅ UPDATE complete. {affected} row(s) changed to status="orphaned".')
    print()
    print('Verify on Telegram:')
    print('  /last5    → ghost rows now hidden')
    print('  /packages → ghost rows now show under refusals with reason "pre-PR-357 ghost"')
    # Only clean up the SSH-key tmpdir AFTER the UPDATE actually
    # ran. A failed UPDATE leaves tmpdir alive too so the operator
    # can fix the issue and re-run without re-running Cell 1.
    shutil.rmtree(tmpdir, ignore_errors=True)


## Rollback

If Cell 4 marked rows you didn't want to orphan, re-run it with this SQL via Cell 2's pattern:

```sql
UPDATE trades SET status = 'open' WHERE status = 'orphaned';
```

(Or filter by the `notes.orphaned_at` timestamp if you only want to roll back this specific migration.)
